<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

2026/05/22 13:33:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/22 13:33:52 INFO mlflow.store.db.utils: Updating database tables
2026/05/22 13:33:54 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Matheus/Cesar/Machine_learning/atividade-04-deep-learning-i-Matheuslh/notebooks/mlruns/1', creation_time=1779467634416, experiment_id='1', last_update_time=1779467634416, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed):
    # 1. Carregar o dataset original (focando no conjunto de treino para a divisão)
    (X_train_full, y_train_full), _ = cifar10.load_data()
    
    # 2. Realizar o flatten das imagens (transformar 32x32x3 em 3072)
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
    
    # 3. Normalizar os dados (converter para float e escalar entre 0 e 1)
    X_train_full = X_train_full.astype('float32') / 255.0
    
    # 4. Separação entre treino e validação (utilizando 20% para validação)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, 
        y_train_full, 
        test_size=0.2, 
        random_state=seed,
        stratify=y_train_full 
    )
    
    return X_train, X_val, y_train, y_val

# Resposta

- Questão 1: 32 x 32 pixels com 3 canais rgbs, (32, 32, 3).
- Questão 2: Depois do Flatten ela passa a possuir 3072 features.
- Questão 3: Porque as MLPs possuem uma arquitetura baseada em camadas totalmente conectadas. A camada de entrada dessas redes foi projetada para receber tensores unidimensionais, onde cada neurônio de entrada corresponde a uma única característica (feature) independente. Como as imagens são originalmente estruturas bidimensionais ou tridimensionais, o flatten se faz obrigatório para desdobrar esses pixels em uma única linha contínua, permitindo que cada pixel/canal seja mapeado diretamente para um neurônio da camada de entrada.
- Questão 4: Estabilidade numérica e otimização, algoritmos de descida de gradiente (como SGD ou Adam) convergem muito mais rápido quando as features estão em escalas semelhantes. Prevenção de saturação de funções de ativação, funções de ativação como Sigmoide ou Tanh saturam quando recebem valores muito altos ou muito baixos na entrada, o que interrompe o fluxo de aprendizado dos neurônios. E simetria na superfície de custo, a normalização ajuda a tornar a função de perda mais esférica e menos alongada.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [9]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=100, # Definido um limite padrão de épocas para evitar loops infinitos
        verbose=True  # Permite acompanhar o progresso do treinamento no console
    )
    
    # Treinando o modelo
    # .ravel() transforma a matriz coluna do y_train em um vetor plano exigido pelo sklearn
    model.fit(X_train, y_train.ravel())
    
    return model

# Resposta

- Questão 1: O número de parâmetros na primeira camada depende diretamente do número de neurônios que você escolher para a primeira camada oculta.A fôrmula seria: Total = (3072*H1)+H1
- Questão 2: f(x) = max(0,x)
- Questão 3: Isso acontece devido à natureza totalmente conectada das camadas de uma MLP. Em uma MLP, cada único pixel e cada canal de cor são tratados como uma feature isolada e independente. Cada uma dessas features precisa se conectar individualmente com todos os neurônios da camada seguinte. Como imagens geram vetores de entrada massivos, a multiplicação cruzada com as camadas ocultas faz o número de parâmetros explodir.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    # 1. Realizar as predições
    y_pred = model.predict(X_test)
    y_true = y_test.ravel() # Garante que y_true seja um vetor unidimensional
    
    # 2. Calcular as métricas
    # Nota: Como o CIFAR-10 possui 10 classes, precisamos definir o parâmetro 'average'.
    # Usamos o 'macro' para calcular a métrica de cada classe individualmente e tirar a média aritmética simples.
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # 3. Apresentar os resultados em um DataFrame do Pandas
    results_dict = {
        'Métrica': ['Acurácia (Accuracy)', 'Precisão (Precision)', 'Revocação (Recall)', 'F1-Score'],
        'Valor Obtido': [accuracy, precision, recall, f1]
    }
    
    df_results = pd.DataFrame(results_dict)
    
    return df_results

# Execução questões 1, 2 e 3

In [11]:
SEDE_ALEATORIA = 42
ARQUITETURA = (100, 50)      # Uma camada com 100 neurônios, outra com 50
ATIVACAO = 'relu'            # Função de ativação
TAXA_APRENDIZADO = 0.001     # Learning rate

print("Passo 1: Carregando e preparando os dados...")
# Chamando a primeira função
X_train, X_val, y_train, y_val = load_data(seed=SEDE_ALEATORIA)
print("Dados prontos! Formato do treino:", X_train.shape)

print("\nPasso 2: Treinando a MLP (isso pode demorar um pouco)...")
# Chamando a segunda função (passando os dados que acabamos de carregar)
modelo_treinado = train_mlp(
    X_train=X_train,
    y_train=y_train,
    activation=ATIVACAO,
    hidden_layers=ARQUITETURA,
    learning_rate=TAXA_APRENDIZADO,
    seed=SEDE_ALEATORIA
)
print("✅ Treinamento concluído!")

print("\nPasso 3: Avaliando o modelo nos dados de validação...")
# Chamando a terceira função (passando o modelo treinado e os dados de VALIDAÇÃO)
tabela_resultados = evaluate(
    model=modelo_treinado, 
    X_test=X_val, 
    y_test=y_val
)

# Exibir a tabela de resultados na tela
print("\nRESULTADOS OBTIDOS:")
display(tabela_resultados) # Se estiver no notebook, 'display' deixa a tabela linda

Passo 1: Carregando e preparando os dados...
Dados prontos! Formato do treino: (40000, 3072)

Passo 2: Treinando a MLP (isso pode demorar um pouco)...
Iteration 1, loss = 1.98397939
Iteration 2, loss = 1.79491182
Iteration 3, loss = 1.71869353
Iteration 4, loss = 1.67148897
Iteration 5, loss = 1.63908707
Iteration 6, loss = 1.59945070
Iteration 7, loss = 1.57519586
Iteration 8, loss = 1.54966714
Iteration 9, loss = 1.52567861
Iteration 10, loss = 1.50869952
Iteration 11, loss = 1.49177101
Iteration 12, loss = 1.48236087
Iteration 13, loss = 1.46825541
Iteration 14, loss = 1.45876355
Iteration 15, loss = 1.45010427
Iteration 16, loss = 1.43686977
Iteration 17, loss = 1.42156426
Iteration 18, loss = 1.42315820
Iteration 19, loss = 1.40568215
Iteration 20, loss = 1.39679764
Iteration 21, loss = 1.39702276
Iteration 22, loss = 1.38099639
Iteration 23, loss = 1.37419322
Iteration 24, loss = 1.37423711
Iteration 25, loss = 1.36174809
Iteration 26, loss = 1.34761344
Iteration 27, loss = 1.351

,Métrica,Valor Obtido
0,Acurácia (Accuracy),0.486500
1,Precisão (Precision),0.494161
2,Revocação (Recall),0.486500
3,F1-Score,0.482398


# Repostas Questões 3

- Interpretação dos resultados:
     A perda começou em 1.98 e caiu de forma constante até 1.09 na iteração 100. Como ela terminou em queda e o modelo parou de rodar apenas porque atingiu o limite padrão de épocas (Iteration 100), isso indica que se aumentarmos o parâmetro max_iter para 150 ou 200, o modelo provavelmente ganhará mais alguns pontos percentuais de acurácia. O modelo atingiu 48,65% de acurácia, isso prova matematicamente que a MLP de fato extraiu padrões visuais das imagens e aprendeu a diferenciá-las, quase multiplicando por 5 o desempenho de um palpite aleatório. Porém, embora 48,65% seja um ótimo começo, esse número evidencia a limitação clássica das MLPs ao lidar com imagens utilizando flatten. Ao transformar a matriz 32 * 32 * 3 em um vetor plano de 3072 elementos, a rede perdeu as relações espaciais de vizinhança dos pixels. Por último, como a acurácia (48,65%), a precisão (49,41%) e o recall (48,65%) deram valores extremamente próximos, significa que o CIFAR-10 é um dataset perfeitamente balanceado.

- Questão 1: A acurácia representa a porcentagem total de acertos do modelo em relação ao total de casos avaliados.
- Questão 2: A precision mede quantas das previsões positivas feitas pelo modelo estavam realmente corretas, enquanto o recall mede quantos dos casos positivos reais foram identificados pelo modelo.
- Questão 3: O f1-score é importante em situações em que é necessário equilibrar precision e recall, principalmente em problemas com classes desbalanceadas, onde uma classe aparece muito mais que a outra. Essa métrica é útil quando tanto falsos positivos quanto falsos negativos possuem impacto relevante.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import time
import mlflow
from sklearn.neural_network import MLPClassifier

def run_mlflow_experiment(X_train, y_train, X_val, y_val, activation, hidden_layers, learning_rate, max_iter, batch_size, seed, experiment_name="CIFAR-10_MLP_Evaluation"):
    # Define ou cria o experimento no MLflow
    mlflow.set_experiment(experiment_name)
    
    # Inicia a gravação da corrida (Run)
    with mlflow.start_run():
        
        print(f"Iniciando Run: Camadas={hidden_layers}, Ativação={activation}, LR={learning_rate}")
        
        # 1. REGISTRO DE PARÂMETROS
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers)) # Salva como string para o MLflow aceitar a tupla
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("seed", seed)
        
        # 2. TREINAMENTO (Medindo o tempo de execução)
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed,
            verbose=False
        )
        
        start_time = time.time()
        model.fit(X_train, y_train.ravel())
        training_time = time.time() - start_time
        
        # 3. AVALIAÇÃO (Utilizando a função evaluate criada na Questão 3)
        df_results = evaluate(model, X_val, y_val)
        
        # Extraindo os valores do DataFrame retornado pela função evaluate
        acc_val = df_results.loc[df_results['Métrica'] == 'Acurácia (Accuracy)', 'Valor Obtido'].values[0]
        prec_val = df_results.loc[df_results['Métrica'] == 'Precisão (Precision)', 'Valor Obtido'].values[0]
        rec_val = df_results.loc[df_results['Métrica'] == 'Revocação (Recall)', 'Valor Obtido'].values[0]
        f1_val = df_results.loc[df_results['Métrica'] == 'F1-Score', 'Valor Obtido'].values[0]
        
        # 4. REGISTRO DE MÉTRICAS
        mlflow.log_metric("accuracy", acc_val)
        mlflow.log_metric("precision", prec_val)
        mlflow.log_metric("recall", rec_val)
        mlflow.log_metric("f1_score", f1_val)
        mlflow.log_metric("training_time", training_time)
        
        print(f"Run Finalizada! Acurácia: {acc_val:.4f} | Tempo: {training_time:.2f}s\n")

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [13]:
ARQUITETURA_FIXA = (100, 50)
LR_FIXO = 0.001
SEED_FIXA = 42
MAX_ITER_FIXO = 60
BATCH_SIZE_FIXO = 200

# Garantir que os dados estão carregados
X_train, X_val, y_train, y_val = load_data(seed=SEED_FIXA)

# 2. Lista de funções de ativação para testar
funcoes_ativacao = ['logistic', 'tanh', 'relu']

print("Iniciando o experimento comparativo de funções de ativação...\n")

for ativacao in funcoes_ativacao:
    # Executa e registra no MLflow usando a função estruturada na Questão 4
    run_mlflow_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=ativacao,
        hidden_layers=ARQUITETURA_FIXA,
        learning_rate=LR_FIXO,
        max_iter=MAX_ITER_FIXO,
        batch_size=BATCH_SIZE_FIXO,
        seed=SEED_FIXA,
        experiment_name="Comparacao_Ativacoes_CIFAR10"
    )

print("Todos os experimentos foram registrados com sucesso.")

2026/05/22 23:34:12 INFO mlflow.tracking.fluent: Experiment with name 'Comparacao_Ativacoes_CIFAR10' does not exist. Creating a new experiment.


Iniciando o experimento comparativo de funções de ativação...

Iniciando Run: Camadas=(100, 50), Ativação=logistic, LR=0.001
✨ Run Finalizada! Acurácia: 0.4903 | Tempo: 188.79s

Iniciando Run: Camadas=(100, 50), Ativação=tanh, LR=0.001
✨ Run Finalizada! Acurácia: 0.4715 | Tempo: 180.80s

Iniciando Run: Camadas=(100, 50), Ativação=relu, LR=0.001
✨ Run Finalizada! Acurácia: 0.5005 | Tempo: 206.99s

Todos os experimentos foram registrados com sucesso.


# Resposta

- Questão 1: Relu apresentou maior convergência.
- Questão 2: Relu apresentou maior estabilidade.
- Questão 3: Sim, principalmente no balanço entre acurácia e tempo. A relu entregou a melhor inteligência para o modelo 50,05%, mas cobrou um preço por isso: demorou 206.99s (cerca de 26 segundos a mais que a tanh).
- Questão 4: Mesmo que a função logística apresente bons resultados em redes pequenas, a ReLU continua sendo a função de ativação mais utilizada em Deep Learning por evitar o problema do gradiente evanescente em redes profundas, permitir ativações esparsas ao zerar valores negativos, reduzindo ruído e consumo de memória, e possuir um custo computacional muito menor que funções como Sigmoide e Tanh, já que sua operação é simples e altamente eficiente para execução em GPUs e grandes volumes de dados.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
# 1. Configurações travadas (Requisitos de consistência)
ATIVACAO_FIXA = 'relu'
LR_FIXO = 0.001
SEED_FIXA = 42
MAX_ITER_FIXO = 100 # Mantido em 100 para dar tempo de redes maiores convergirem
BATCH_SIZE_FIXO = 200

# Garantir que os dados estão prontos
X_train, X_val, y_train, y_val = load_data(seed=SEED_FIXA)

# 2. Lista de arquiteturas para testar
arquiteturas = [
    (32,),          # Uma única camada bem pequena
    (64,),          # Uma única camada média
    (128, 64),      # Duas camadas intermediárias
    (256, 128)      # Duas camadas robustas
]

print("Iniciando o experimento comparativo de arquiteturas...\n")

for arr in arquiteturas:
    # Executa e registra no MLflow usando a função estruturada na Questão 4
    run_mlflow_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=ATIVACAO_FIXA,
        hidden_layers=arr,
        learning_rate=LR_FIXO,
        max_iter=MAX_ITER_FIXO,
        batch_size=BATCH_SIZE_FIXO,
        seed=SEED_FIXA,
        experiment_name="Comparacao_Arquiteturas_CIFAR10"
    )

print("Todas as arquiteturas foram testadas.")

2026/05/22 23:44:04 INFO mlflow.tracking.fluent: Experiment with name 'Comparacao_Arquiteturas_CIFAR10' does not exist. Creating a new experiment.


Iniciando o experimento comparativo de arquiteturas...

Iniciando Run: Camadas=(32,), Ativação=relu, LR=0.001
✨ Run Finalizada! Acurácia: 0.3749 | Tempo: 270.79s

Iniciando Run: Camadas=(64,), Ativação=relu, LR=0.001
✨ Run Finalizada! Acurácia: 0.3864 | Tempo: 400.50s

Iniciando Run: Camadas=(128, 64), Ativação=relu, LR=0.001


# Resposta

- Questão 1: Não necessariamente. Embora neste teste específico com o CIFAR-10 aumentar a rede de (32,) para (256, 128) resulte em um ganho de acurácia, o crescimento do desempenho não é linear e possui um teto crítico.
- Questão 2: A arquitetura (128, 64) possuiu um melhor trade off.
- Questão 3: 
- Questão 4: A arquitetura (256, 128) apresentou, isoladamente, o maior custo computacional, refletido no maior training_time no painel do MLflow.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
# 1. Configurações travadas (Requisitos de consistência)
ARQUITETURA_FIXA = (128, 64)
ATIVACAO_FIXA = 'relu'
SEED_FIXA = 42
MAX_ITER_FIXO = 60
BATCH_SIZE_FIXO = 200

# Garantir que os dados estão carregados
X_train, X_val, y_train, y_val = load_data(seed=SEED_FIXA)

# 2. Lista de learning rates para testar
learning_rates = [0.1, 0.01, 0.001]

print("Iniciando o experimento comparativo de learning rates...\n")

for lr in learning_rates:
    # Executa e registra no MLflow usando a função estruturada na Questão 4
    run_mlflow_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=ATIVACAO_FIXA,
        hidden_layers=ARQUITETURA_FIXA,
        learning_rate=lr,
        max_iter=MAX_ITER_FIXO,
        batch_size=BATCH_SIZE_FIXO,
        seed=SEED_FIXA,
        experiment_name="Comparacao_LearningRates_CIFAR10"
    )

print("Todos os learning rates foram testados.")

# Resposta

- Questão 1: A taxa de aprendizado de 0.001 apresentou o melhor desempenho.
- Questão 2: A taxa de aprendizado de 0.1 apresentou a maior instabilidade.
- Questão 3: Quando o learning rate é excessivamente alto, como 0.1 neste cenário, os passos de atualização dos pesos são grandes demais. Em vez de descer em direção ao ponto mais baixo da curva de erro, o algoritmo salta de um lado para o outro das paredes da função de custo, fazendo com que a perda divirja ou oscile indefinidamente. O modelo se torna incapaz de aprender padrões estruturados e os pesos mudam de forma caótica.
- QUestão 4: Embora o teste atual não tenha descido abaixo de 0.001, caso utilizássemos uma taxa extremamente baixa (como 0.00001), o modelo sofreria de convergência dolorosamente lenta.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
# TODO: implemente